# Phase 2: Baseline NLP Models

This notebook reviews classical NLP baselines for complaint product classification. The training pipeline uses `text_ml_clean` as input and predicts the CFPB `Product` category.

Models covered:

- DummyClassifier baseline
- Bag of Words + Logistic Regression
- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cfpb_sample_90k_clean.csv'
RESULTS_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / 'baseline_model_results.csv'
BEST_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'tfidf_logistic_regression.joblib'


## Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, usecols=['text_ml_clean', 'target'], low_memory=False)
df.shape


In [ ]:
df['target'].value_counts()


## Baseline Results

In [ ]:
results = pd.read_csv(RESULTS_PATH)
results.sort_values('macro_f1', ascending=False)


In [ ]:
plt.figure(figsize=(9, 4))
ordered = results.sort_values('macro_f1', ascending=True)
sns.barplot(data=ordered, x='macro_f1', y='model_name')
plt.title('Baseline Model Macro F1')
plt.xlabel('Macro F1')
plt.ylabel('Model')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


## Best Model Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text_ml_clean'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target'],
)
model = joblib.load(BEST_MODEL_PATH)
preds = model.predict(X_test)
print(classification_report(y_test, preds, zero_division=0))


In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, preds, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('TF-IDF Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## Error Analysis

In [ ]:
errors = pd.DataFrame({
    'text': X_test.reset_index(drop=True),
    'actual': y_test.reset_index(drop=True),
    'predicted': pd.Series(preds),
})
errors = errors[errors['actual'] != errors['predicted']]
errors.shape


In [ ]:
errors.groupby(['actual', 'predicted']).size().sort_values(ascending=False).head(15)


In [ ]:
errors.sample(10, random_state=42)
